# Population Density Mapping

## 📊 Business Context
Visualize demographic shifts.

**Analytical Approach:** Raster Risk
This notebook utilizes advanced analytics to derive actionable insights.

In [ ]:
# Import Libraries
import ee
import geemap
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Initialize Earth Engine
try:
    ee.Initialize()
except:
    ee.Authenticate()
    ee.Initialize()

In [ ]:
# Define Area of Interest (AOI)
# Using a sample coordinate (e.g., Lagos, Nigeria or similar)
AOI = ee.Geometry.Point([3.3792, 6.5244]).buffer(20000)

def analyze_risk():
    print('Loading datasets...')
    # 1. Elevation (SRTM)
    srtm = ee.Image('USGS/SRTMGL1_003').clip(AOI)
    elevation = srtm.select('elevation')
    slope = ee.Terrain.slope(elevation)
    
    # 2. Land Cover (ESA WorldCover)
    landcover = ee.ImageCollection('ESA/WorldCover/v100').first().clip(AOI)
    
    # 3. Risk Calculation Logic
    # Example: Flood Risk = Low Elevation (< 10m) AND Flat Slope (< 5 deg)
    low_elevation = elevation.lt(10)
    flat_slope = slope.lt(5)
    
    # Weighted Overlay
    risk_score = low_elevation.multiply(0.6).add(flat_slope.multiply(0.4))
    high_risk = risk_score.gt(0.8)
    
    # Calculate Risk Area
    area_image = high_risk.multiply(ee.Image.pixelArea())
    stats = area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=AOI,
        scale=30,
        maxPixels=1e9
    )
    risk_area_sqkm = stats.get('elevation').getInfo() / 1e6
    print(f'Estimated High Risk Area: {risk_area_sqkm:.2f} sq km')
    
    # Visualization
    m = geemap.Map(center=[6.5244, 3.3792], zoom=11)
    
    # Vis Params
    elev_vis = {'min': 0, 'max': 100, 'palette': ['blue', 'green', 'yellow', 'brown']}
    risk_vis = {'min': 0, 'max': 1, 'palette': ['white', 'red']}
    
    m.addLayer(elevation, elev_vis, 'Elevation')
    m.addLayer(high_risk.updateMask(high_risk), {'palette': ['red']}, 'High Flood Risk Zone')
    
    m.add_colorbar(elev_vis, label='Elevation (m)')
    return m

m = analyze_risk()
m

## 🗺️ Spatial Insights

1. **Risk Hotspots**: The red areas on the map indicate regions with high susceptibility to the analyzed risk factor (e.g., flooding).
2. **Topography**: Low-lying areas (blue/green) are naturally more vulnerable.
3. **Mitigation**: Urban planning in these zones should prioritize drainage systems and flood barriers.